# Run-book — build the test AML images for one commit

**What this proves:** both Azure ML node images exist and are built from one pinned rise-eo commit,
under the **test** names `rise-eo-aml-env-test` and `rise-eo-infer-sklearn-test`. Other run-books
then put the printed `rise-eo-aml-env-test:<version>` in their `AZ_ENV` setting. Test runs use these
names so the formal images (`rise-eo-aml-env`, `rise-eo-infer-sklearn`) gain a version only at a
release (#234).

**Claude does not run this** (spec 24). You run it cell by cell and paste back the **Summary**
cell's output, or a failing cell's message. Every check is a plain `assert` you can read: the
text above a check cell says what PASS means, and the `assert` says the same thing in code.

**Needs:** an Azure login that `~/.config/rise-eo/config.toml` describes, including
`image_registry` (`rise_eo.config.load()`); the commit pushed to GitHub. Time (estimates): seconds
when an image for this commit already exists; otherwise 10–20 minutes per image while ACR builds it.

## Before you start

**Kernel — which Python runs the cells.** Use **`.venv`**: the Python at `rise-eo/.venv/bin/python`
in the **main** checkout (Python 3.11). It is the only environment with rise-eo's dependencies.
- Open VS Code on the **`rise-eo/` folder** (the main checkout) — even when this notebook lives in a
  worktree under `rise-eo/.claude/worktrees/<name>/`. Then **Select Kernel** (top right) →
  **Python Environments** → **`.venv`**.
- Not listed (e.g. VS Code was opened on the worktree folder, which has no `.venv` of its own)?
  **Select Another Kernel… → Python Environments → Enter interpreter path…** (wording varies by
  VS Code version) and give the main checkout's `rise-eo/.venv/bin/python`.
- The **Setup** cell below checks this and stops with a message if the kernel is wrong.

**Code — which `rise-eo` gets imported.** rise-eo is installed from the *main* checkout, so inside a
worktree a plain `import rise_eo` would quietly run `main`'s code, not the branch under test. The
Setup cell puts **this notebook's own** `src/` first on Python's import path and checks it worked.
(This is what the old run-books did with `export PYTHONPATH=src` — the same thing, invisibly.)

**Stop / resume.** Interrupt the kernel (■) to stop. Re-running Step 1 resumes: it reuses any image
already registered for this commit and builds nothing again. After a kernel restart, run **Setup** and **Settings** again,
then continue at any step — every step reads what it needs from disk.

**When you are done.** Paste back what the **Summary** cell says to paste (plus any failing cell's
message). Then **Restart & Clear All Outputs** and save: this file is tracked in git, and a test
refuses saved outputs.

In [ ]:
# Setup -- which Python, and which rise-eo. Nothing to change here.
import sys
from pathlib import Path

# 1. Which Python? It must be rise-eo's own virtual environment, rise-eo/.venv.
VENV = Path(sys.prefix)
assert VENV.name == ".venv" and (VENV.parent / "pyproject.toml").exists(), (
    f"Wrong kernel: {sys.prefix} is not rise-eo's .venv. Pick the '.venv' kernel (see above).")
MAIN_CHECKOUT = VENV.parent              # the rise-eo checkout that owns the venv

# 2. Which rise-eo? The one next to this notebook: <checkout>/runbooks/<this> -> <checkout>/src.
#    Jupyter and VS Code start the kernel in the notebook's own folder.
CHECKOUT = Path.cwd().parent
assert Path.cwd().name == "runbooks" and (CHECKOUT / "src" / "rise_eo").is_dir(), (
    f"The kernel started in {Path.cwd()}, not in this notebook's runbooks/ folder.")
if str(CHECKOUT / "src") not in sys.path:
    sys.path.insert(0, str(CHECKOUT / "src"))
import rise_eo

assert Path(rise_eo.__file__).is_relative_to(CHECKOUT / "src"), (
    f"rise-eo was already imported from {rise_eo.__file__}. Restart the kernel, then run this cell first.")
print("Python :", sys.executable)
print("rise-eo    :", rise_eo.__file__)

## Settings

Every input this run-book uses. Change them here and nowhere else.

In [ ]:
# Settings
import json
import re

import rise_eo
import rise_eo.aml
from rise_eo.image import ImageDefinition
from rise_eo.image import registry as image_registry

cfg = rise_eo.config.load()      # resource group, workspace, image registry: private, read at run time

# The full 40-character sha of the commit to build: `git rev-parse origin/main` for the latest main.
COMMIT = "<40-character sha>"
assert re.fullmatch(r"[0-9a-f]{40}", COMMIT), "set COMMIT to a full 40-character sha"

# Test names: never the formal rise-eo-aml-env / rise-eo-infer-sklearn, and no alias (#234).
BASE_NAME = "rise-eo-aml-env-test"
INFER_NAME = "rise-eo-infer-sklearn-test"
RISE_EO_REF = f"git+https://github.com/nasaharvest/rise-eo@{COMMIT}"   # a git ref, so pip records the commit

OUT = MAIN_CHECKOUT / "tests" / "outputs" / "test_images" / COMMIT[:12]   # gitignored
OUT.mkdir(parents=True, exist_ok=True)

## Step 1 — ensure both images

**What it does:** declares both images from `RISE_EO_REF` and calls `rise_eo.aml.ensure_environment`
for each. When an image for this exact definition is already registered and still exists in Azure ML,
it is reused and nothing is built. Otherwise the call registers a new version and ACR starts building
it in the background.

**PASS if:** both images report `reused`. On a first run the cell stops at the first image that was just
built and prints its Studio link. Open the link, wait for **Build status: Succeeded**, then re-run the
cell. It now finds that image (reused) and moves on to the next one.
`reused` means the image is registered, not that it built: only the Studio page shows
**Build status: Succeeded** (#236), so check it for every new build before you run another
run-book.

**Time:** seconds per reused image; 10–20 minutes per new build (you wait in Studio, not here).

In [ ]:
# Step 1 -- ensure both environments; a new build stops here until Studio says Succeeded.
BASE = ImageDefinition(name=BASE_NAME, rise_eo=RISE_EO_REF, extras=("local", "azure", "mpc"))
INFER = BASE.derive(name=INFER_NAME, extra_pip=("scikit-learn", "joblib"))

envs = {}
for key, defn in (("aml_env", BASE), ("infer_env", INFER)):
    env = rise_eo.aml.ensure_environment(
        defn, registry=cfg.image_registry,
        resource_group=cfg.resource_group, workspace=cfg.workspace, storage="azure",
    )
    print(key, env.ref, "reused" if env.reused else "JUST BUILT")
    assert env.reused, (
        f"{env.ref} was just built: open {env.build_url} , wait for `Build status: Succeeded`, "
        "then re-run this cell.")
    envs[key] = {"ref": env.ref, "registry_ref": env.registry_ref}
(OUT / "envs.json").write_text(json.dumps(envs, indent=2))
print("written:", OUT / "envs.json")

## Step 2 — check what was registered

**What it does:** reads `envs.json` back from disk and, for each image, the definition the image
registry recorded.

**PASS if:** both names are the test names and both definitions install rise-eo from exactly
`RISE_EO_REF`.

In [ ]:
# Step 2 -- read envs.json and the registry's recorded definitions back, then assert.
envs = json.loads((OUT / "envs.json").read_text())
assert envs["aml_env"]["ref"].startswith(f"{BASE_NAME}:"), envs["aml_env"]["ref"]
assert envs["infer_env"]["ref"].startswith(f"{INFER_NAME}:"), envs["infer_env"]["ref"]
for key, env in envs.items():
    resolved = image_registry.resolve(env["registry_ref"], cfg.image_registry)
    assert resolved.definition.get("rise_eo") == RISE_EO_REF, (
        f"{key}: the registry says rise_eo={resolved.definition.get('rise_eo')!r}, expected {RISE_EO_REF!r}")
print("PASS: both images are test images built from", COMMIT)

## Summary — paste this output back

Re-reads everything from disk, so it works after a restart. Put `AZ_ENV` into the Settings of the
run-book you run next.

In [ ]:
import json

envs = json.loads((OUT / "envs.json").read_text())
summary = {
    "commit": COMMIT,
    "AZ_ENV": envs["aml_env"]["ref"],
    "inference_env": envs["infer_env"]["ref"],
}
print(json.dumps(summary, indent=2))